In [1]:
import sys
sys.path.append('..')

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=r"C:\Users\IvaRichterová\Desktop\entsoe-dashboard\.env")

cache_slozka = r"C:\Users\IvaRichterová\Desktop\entsoe-dashboard\data\cache"

# Načteme vše z Parquet
ceny_cz = pd.read_parquet(os.path.join(cache_slozka, "ceny_CZ.parquet"))
ceny_cz = ceny_cz.iloc[:, 0]

load_actual = pd.read_parquet(os.path.join(cache_slozka, "load_actual.parquet"))
load_forecast = pd.read_parquet(os.path.join(cache_slozka, "load_forecast.parquet"))

vyroba = pd.read_parquet(os.path.join(cache_slozka, "vyroba_cz.parquet"))

print("✓ Vše načteno z cache")
print(f"  Ceny: {len(ceny_cz)} hodnot")
print(f"  Load actual: {len(load_actual)} hodnot")
print(f"  Výroba: {vyroba.shape}")

✓ Vše načteno z cache
  Ceny: 673 hodnot
  Load actual: 2876 hodnot
  Výroba: (2876, 16)


In [2]:
# Stejná úprava jako v notebooku 03
vyroba_clean = vyroba.xs("Actual Aggregated", axis=1, level=1)

nazvy = {
    "Biomass": "Biomasa",
    "Fossil Brown coal/Lignite": "Hnědé uhlí",
    "Fossil Coal-derived gas": "Uhelný plyn",
    "Fossil Gas": "Zemní plyn",
    "Fossil Hard coal": "Černé uhlí",
    "Fossil Oil": "Ropa",
    "Hydro Pumped Storage": "Přečerpávací voda",
    "Hydro Run-of-river and poundage": "Průtočná voda",
    "Hydro Water Reservoir": "Vodní nádrž",
    "Nuclear": "Jádro",
    "Other": "Ostatní",
    "Other renewable": "Ostatní OZE",
    "Solar": "Solár",
    "Waste": "Odpad",
    "Wind Onshore": "Vítr",
}

vyroba_clean = vyroba_clean.rename(columns=nazvy)

vyroba_clean["OZE celkem"] = (
    vyroba_clean["Solár"] +
    vyroba_clean["Vítr"] +
    vyroba_clean["Biomasa"] +
    vyroba_clean["Průtočná voda"] +
    vyroba_clean["Ostatní OZE"]
)

vyroba_clean["Celkem"] = vyroba_clean.sum(axis=1)
vyroba_clean["Podíl OZE %"] = (vyroba_clean["OZE celkem"] / vyroba_clean["Celkem"] * 100).round(1)

print("✓ Výrobní data připravena")

✓ Výrobní data připravena


In [3]:
print("Sloupce výroby:")
print(vyroba.columns.tolist())

Sloupce výroby:
[('Biomass', 'Actual Aggregated'), ('Fossil Brown coal/Lignite', 'Actual Aggregated'), ('Fossil Coal-derived gas', 'Actual Aggregated'), ('Fossil Gas', 'Actual Aggregated'), ('Fossil Hard coal', 'Actual Aggregated'), ('Fossil Oil', 'Actual Aggregated'), ('Hydro Pumped Storage', 'Actual Aggregated'), ('Hydro Pumped Storage', 'Actual Consumption'), ('Hydro Run-of-river and poundage', 'Actual Aggregated'), ('Hydro Water Reservoir', 'Actual Aggregated'), ('Nuclear', 'Actual Aggregated'), ('Other', 'Actual Aggregated'), ('Other renewable', 'Actual Aggregated'), ('Solar', 'Actual Aggregated'), ('Waste', 'Actual Aggregated'), ('Wind Onshore', 'Actual Aggregated')]


In [4]:
# Insight 1: Počet hodin kdy cena byla nad 100 EUR/MWh
hodiny_drazsi = (ceny_cz > 100).sum()
print(f"Hodiny s cenou nad 100 EUR/MWh: {hodiny_drazsi}")

# Insight 2: Hodina s nejvyšším podílem OZE
max_oze_cas = vyroba_clean["Podíl OZE %"].idxmax()
max_oze_hodnota = vyroba_clean["Podíl OZE %"].max()
print(f"Nejvyšší podíl OZE: {max_oze_hodnota:.1f}% v {max_oze_cas.strftime('%d.%m %H:%M')}")

# Insight 3: Největší odchylka forecast vs actual
odchylka = load_actual["Actual Load"] - load_forecast["Forecasted Load"]
max_odchylka_cas = odchylka.abs().idxmax()
max_odchylka_hodnota = odchylka[max_odchylka_cas]
print(f"Největší odchylka load: {max_odchylka_hodnota:.0f} MW v {max_odchylka_cas.strftime('%d.%m %H:%M')}")

# Insight 4: Průměrná cena při vysokém vs nízkém podílu OZE
ceny_reindexed = ceny_cz.reindex(vyroba_clean.index, method="nearest")
vysoke_oze = ceny_reindexed[vyroba_clean["Podíl OZE %"] > 25].mean()
nizke_oze = ceny_reindexed[vyroba_clean["Podíl OZE %"] < 10].mean()
print(f"Průměrná cena při OZE >25%: {vysoke_oze:.1f} EUR/MWh")
print(f"Průměrná cena při OZE <10%: {nizke_oze:.1f} EUR/MWh")

Hodiny s cenou nad 100 EUR/MWh: 412
Nejvyšší podíl OZE: 30.1% v 13.03 11:45
Největší odchylka load: 492 MW v 01.03 15:00
Průměrná cena při OZE >25%: 14.4 EUR/MWh
Průměrná cena při OZE <10%: 40.0 EUR/MWh


In [5]:
import os

export_slozka = r"C:\Users\IvaRichterová\Desktop\entsoe-dashboard\exports"
os.makedirs(export_slozka, exist_ok=True)

print("✓ Insights nemají grafy k exportu — výsledky jsou textové")


✓ Insights nemají grafy k exportu — výsledky jsou textové
